# 🔎 Aula 08 — Embeddings, Similaridade e RAG
## Guilda de IA — Introdução à IA Generativa

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/aula08_embeddings_rag_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Objetivos:**
1. Embeddings: texto como vetores
2. FAISS: vector store em memória
3. Busca semântica: similaridade por cosseno
4. Agente RAG: LLM com ferramenta de busca

**Modelos:**
- `gemma4:e2b-it-qat` — LLM (Gemma 4 E2B com QAT, suporta tool calling)
- `nomic-embed-text-v2-moe` — modelo de embedding (Matryoshka, 256 dim)

## 🏗️ Setup

⚠️ Vá em `Runtime → Change runtime type` → **T4 GPU**

Execute a célula abaixo e aguarde — leva ~2 min na primeira vez.
Vamos carregar dois modelos que coexistem na GPU T4:
- **gemma4:e2b-it-qat** (~2B params, QAT) — o LLM
- **nomic-embed-text-v2-moe** (~0.5B params) — o modelo de embedding

In [ ]:
# ── Setup: Ollama + gemma4:e2b-it-qat + nomic-embed-text-v2-moe + deps ────
# Tudo em uma célula. Execute e prossiga.

# 1. Instalar Ollama + deps Python
!apt-get install -y zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q langchain langchain-openai langchain-core langgraph faiss-cpu sentence-transformers nest_asyncio requests

# 2. Workaround GPU Colab + keep alive
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

# 3. Iniciar servidor
!pkill -f ollama 2> /dev/null; sleep 1
import subprocess
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env={**os.environ})

# 4. Aguardar servidor
import time, requests
for i in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            break
    except:
        time.sleep(1)

# 5. Baixar modelos
!ollama pull gemma4:e2b-it-qat
!ollama pull nomic-embed-text-v2-moe

# 6. Warm up LLM
print("🔥 Warm up LLM...")
start = time.time()
!curl -s http://localhost:11434/api/chat -d '{"model":"gemma4:e2b-it-qat","messages":[{"role":"user","content":"Hi"}],"stream":false,"keep_alive":-1}' > /dev/null
print(f"✅ LLM pronto em {time.time()-start:.1f}s")

# 7. Warm up embedding model
print("🔥 Warm up embedding model...")
start = time.time()
!curl -s http://localhost:11434/api/embeddings -d '{"model":"nomic-embed-text-v2-moe","prompt":"warm up","keep_alive":-1}' > /dev/null
print(f"✅ Embedding model pronto em {time.time()-start:.1f}s")
print("\n🎉 Tudo pronto! GPU T4 com 2 modelos carregados.")

## 1. Banco de fatos

Vamos criar um pequeno banco de fatos sobre **filosofia e computação**.
Esses fatos são curtos (cabem em 512 tokens) e alguns se complementam semanticamente
— perfeito para testar busca semântica e combinação de informações.

In [ ]:
# ── Banco de fatos: filosofia + computação ──────────────────────────
# Fatos curtos, autocontidos, extraídos do zettelkasten ~/git/biblio.
# Clusters semanticamente próximos para testar busca semântica:
#   - Computação / indecidibilidade / teoria da computação
#   - Filosofia da mente / consciência / cognição
#   - Filosofia clássica / metafísica / identidade
# Conexões inter-cluster: Church-Turing (comp↔cog), Searle (IA↔semântica),
#   mente estendida (cog↔ferramentas), Hofstadter (Gödel↔consciência)

fatos = [
    # ── Computação / Indecidibilidade ──
    "Alan Turing publicou 'On Computable Numbers, with an Application to the Entscheidungsproblem' em 1936, introduzindo a Máquina de Turing como modelo formal de algoritmo e demonstrando a indecidibilidade do problema da decisão.",
    "A Tese de Church-Turing afirma que qualquer processo cognitivo efetivo realizável por uma sequência finita de passos pode ser simulado por uma Máquina de Turing; é um dos fundamentos da ciência cognitiva computacional.",
    "O Problema da Parada é indecidível: nenhuma Máquina de Turing pode decidir, para toda entrada, se uma MT arbitrária para ou entra em loop infinito. A prova usa diagonalização, técnica relacionada ao teorema de Gödel.",
    "O Teorema de Rice (1953) estabelece que toda propriedade não-trivial de linguagens recursivamente enumeráveis é indecidível — por exemplo, decidir se uma MT aceita linguagem vazia, finita ou regular.",
    "Kurt Gödel demonstrou em 1931 seus teoremas de incompletude: em qualquer sistema formal consistente suficiente para a aritmética, existem proposições verdadeiras indecidíveis dentro do sistema.",
    "David Hilbert apresentou em 1900 uma lista de 23 problemas matemáticos; o 10º pedia um algoritmo para decidir solubilidade de equações diofantinas, mostrando-se indecidível via Máquina de Turing.",
    "A Hierarquia de Chomsky classifica linguagens formais em quatro tipos (regular, livre de contexto, sensível ao contexto, recursivamente enumerável), cada uma reconhecida por um autômato de poder crescente até a Máquina de Turing.",
    "John von Neumann propôs em 1948 modelos de autômatos celulares e máquinas autorreplicadoras, base teórica da computação celular e da artificial life.",
    "Warren McCulloch e Walter Pitts publicaram em 1943 'A Logical Calculus of the Ideas Immanent in Nervous Activity', estabelecendo fundamentos matemáticos das redes neurais artificiais.",
    "Claude Shannon fundou a Teoria da Informação em 1948 ('A Mathematical Theory of Communication'), quantificando informação via entropia e estabelecendo limites da compressão e transmissão de dados.",

    # ── Filosofia da mente / Consciência / Cognição ──
    "John Searle propôs em 1980 o experimento mental da 'Sala Chinesa', argumentando que uma máquina pode manipular símbolos sintaticamente sem compreensão semântica, criticando a IA forte.",
    "Andy Clark e David Chalmers publicaram 'The Extended Mind' em 1998, defendendo o Princípio da Paridade: se um processo externo funcionasse na cabeça seria cognitivo, então é cognitivo mesmo fora dela (exemplo do caderno de Otto).",
    "Francisco Varela, Evan Thompson e Eleanor Rosch publicaram 'The Embodied Mind' (1991), fundando o enativismo: cognição como ação situada corporificada, não processamento interno de representações.",
    "Humberto Maturana e Francisco Varela propuseram o conceito de autopoiese: sistemas vivos se auto-produzem e mantêm fechamento organizacional, base da biologia do conhecer ('Árvore do Conhecimento', 1987).",
    "Douglas Hofstadter introduziu o conceito de 'strange loop' em 'Gödel, Escher, Bach' (1979) e o aprofundou em 'I Am a Strange Loop' (2007), descrevendo o 'Eu' como padrão autorreferencial distribuído em múltiplos substratos.",
    "Ludwig Wittgenstein, nas 'Investigações Filosóficas' (1953), propôs que o significado de uma palavra é seu uso na linguagem (jogos de linguagem) e argumentou contra a linguagem privada com o argumento do 'besouro na caixa'.",
    "Daniel Dennett defende que o sujeito é um 'sistema intencional' e não uma substância pensante, crítica ao cartesianismo presente em obras como 'Consciousness Explained' (1991).",
    "Shaun Gallagher, em 'Enactivist Interventions' (2017), argumenta que a memória não é reviver o passado, mas uma descrição construída no presente — perspectiva enativa anti-representacional.",

    # ── Filosofia clássica / Metafísica / Identidade ──
    "Aristóteles, na Metafísica (Livro III), enumera as quatro causas — material, formal, motriz e final — como modos de explicar o ser e a mudança, base da ontologia clássica.",
    "Baruch Spinoza, na Ética (1677), defende que existe uma única substância infinita — Deus sive Natura — e que mente e corpo são atributos paralelos da mesma substância, negando o dualismo cartesiano.",
    "René Descartes, nas Meditações Metafísicas (1641), formula o dualismo substancial mente-corpo e o 'cogito ergo sum' como fundamento do conhecimento certo.",
    "Platão, na República, expõe a maiêutica socrática — método de descoberta da verdade por questionamentos sucessivos — e a alegoria da caverna sobre o conhecimento das Formas.",
    "Gilles Deleuze, em 'Diferença e Repetição' (1968), critica a representação e propõe uma ontologia da diferença pura, influenciando a filosofia da imanência e o pensamento não-representacional.",
]

print(f"📚 {len(fatos)} fatos carregados")
print("=" * 60)
for i, fato in enumerate(fatos):
    print(f"  [{i:02d}] {fato[:80]}...")

## 2. Embeddings com nomic-embed-text-v2-moe

Vamos converter cada fato em um vetor de **256 dimensões** usando o `nomic-embed-text-v2-moe`.

Este modelo suporta **Matryoshka embeddings** — podemos truncar para 768, 512, 256, 128 dims.
Vamos usar 256: leve e suficiente para nossa demo.

In [ ]:
# ── Gerar embeddings via Ollama API ───────────────────────────────
import requests
import numpy as np

EMBED_DIM = 256  # Matryoshka: truncar em 256 dims
OLLAMA_URL = "http://localhost:11434/api/embeddings"

def gerar_embedding(texto: str, dim: int = EMBED_DIM) -> np.ndarray:
    """Gera embedding via Ollama e trunca para `dim` dimensões (Matryoshka)."""
    resp = requests.post(OLLAMA_URL, json={
        "model": "nomic-embed-text-v2-moe",
        "prompt": texto,
        "keep_alive": -1,
    })
    emb = resp.json()["embedding"]
    emb = np.array(emb[:dim], dtype=np.float32)  # truncar (Matryoshka!)
    emb = emb / np.linalg.norm(emb)  # normalizar (L2) para similaridade por cosseno
    return emb

print(f"🔄 Gerando embeddings ({EMBED_DIM} dims) para {len(fatos)} fatos...")
vetores = np.array([gerar_embedding(f) for f in fatos])
print(f"✅ Shape: {vetores.shape} — {vetores.nbytes / 1024:.1f} KB em memória")
print(f"\nExemplo — fato 0:")
print(f"  Texto: {fatos[0][:60]}...")
print(f"  Vetor: [{vetores[0][0]:.4f}, {vetores[0][1]:.4f}, {vetores[0][2]:.4f}, ...]")

## 3. Vector Store com FAISS

FAISS (Facebook AI Similarity Search) é uma biblioteca da Meta para busca de
similaridade em vetores densos. É rápida, simples, e funciona em memória.

Como normalizamos os vetores (L2), a busca por distância L2 é equivalente
a similaridade por cosseno.

In [ ]:
# ── Criar índice FAISS ─────────────────────────────────────────────
import faiss

index = faiss.IndexFlatL2(EMBED_DIM)  # índice plano, distância L2
index.add(vetores)  # adicionar todos os vetores

print(f"✅ Índice FAISS criado: {index.ntotal} vetores de {EMBED_DIM} dimensões")
print(f"   Tipo: IndexFlatL2 (busca exata, força bruta)")
print(f"   Em memória — ideal para demos e datasets pequenos")

## 4. Busca semântica

Agora vamos buscar! Fazemos uma pergunta, geramos o embedding da pergunta,
e o FAISS encontra os fatos mais próximos no espaço vetorial.

In [ ]:
# ── Busca semântica ────────────────────────────────────────────────
def buscar(pergunta: str, k: int = 3) -> list[tuple[str, float]]:
    """Busca os k fatos mais similares à pergunta."""
    query_emb = gerar_embedding(pergunta).reshape(1, -1)
    distancias, indices = index.search(query_emb, k)

    resultados = []
    for i, (dist, idx) in enumerate(zip(distancias[0], indices[0])):
        # distância L2 → similaridade (0 = idêntico, maior = mais distante)
        sim = 1 - dist / 2  # conversão aproximada para [0, 1]
        resultados.append((fatos[idx], sim, idx))
    return resultados

# ── Demo 1: pergunta simples ──
pergunta = "Quem trabalhou com lógica e pensamento?"
print(f"❓ Pergunta: {pergunta}")
print("=" * 60)

for fato, sim, idx in buscar(pergunta, k=3):
    print(f"  [{sim:.3f}] #{idx:02d}: {fato}")
    print()

In [ ]:
# ── Demo 2: pergunta que conecta filosofia + computação ──
pergunta2 = "Como linguagem e significado se relacionam?"
print(f"❓ Pergunta: {pergunta2}")
print("=" * 60)

for fato, sim, idx in buscar(pergunta2, k=3):
    print(f"  [{sim:.3f}] #{idx:02d}: {fato}")
    print()

## 5. Agente RAG: LLM com ferramenta de busca

Agora a parte mais legal: criamos um **agente** com o `gemma4:e2b-it-qat`
e damos a ele uma **ferramenta** para buscar fatos no nosso banco.

O agente recebe uma pergunta, decide *se* precisa buscar, *o que* buscar,
e combina os fatos encontrados para responder.

⚠️ Nomes e descrições em **inglês** — modelos entendem tool calling melhor em inglês.

In [ ]:
# ── Agente RAG com ferramenta de busca ─────────────────────────────
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

@tool
def search_facts(query: str, k: int = 3) -> str:
    """Search a knowledge base of facts about philosophy and computer science.
    Returns the k most semantically similar facts to the query.
    Use this tool when you need to find information about philosophers,
    computer scientists, logic, consciousness, or the history of computing.
    You can call this tool multiple times with different queries to gather
    information from multiple angles before answering.

    Args:
        query: A natural language question or topic to search for.
        k: Number of facts to retrieve (default 3, max 5).
    """
    k = min(k, 5)
    resultados = buscar(query, k=k)
    if not resultados:
        return "No facts found."
    saida = []
    for i, (fato, sim, idx) in enumerate(resultados, 1):
        saida.append(f"Fact {i} (similarity {sim:.2f}): {fato}")
    return "\n\n".join(saida)

llm = ChatOpenAI(
    model="gemma4:e2b-it-qat",
    base_url="http://localhost:11434/v1",
    api_key="nao_precisa",
    temperature=0,
)

# ── System prompt: FORÇA o agente a sempre buscar ──────────────────
# O ponto pedagógico do RAG: o LLM NÃO deve confiar no conhecimento
# armazenado nos seus pesos. Deve SEMPRE buscar fatos externos primeiro.
# Sem isso, o modelo pode "alucinar" ou responder de memória —
# exatamente o que o RAG existe pra evitar.
#
# Análise RCEF-TC (framework da Aula 02):
#   R (Role)       ✅ "You are a RAG agent..."
#   C (Context)    ✅ "...facts about philosophy and computer science"
#   E (Examples)   ❌ cortado — Gemma 4 E2B é pequeno, system prompt
#                   longo prejudica a performance. Num modelo maior
#                   (30B+), valeria a pena adicionar exemplos de
#                   buscas boas vs ruins.
#   F (Format)     ❌ cortado pela mesma razão — num modelo maior,
#                   pediríamos resposta estruturada com citações.
#   T (Task)       ✅ "ALWAYS use search_facts before answering"
#   C (Constraints)✅ "NEVER answer from your own knowledge" + 5 regras

SYSTEM_PROMPT = """You are a RAG agent with access to a knowledge base of facts about
philosophy and computer science. You MUST follow these rules:

1. ALWAYS use the search_facts tool before answering. NEVER answer
   from your own knowledge — your parametric knowledge is NOT trusted.
2. If the question involves multiple topics, call search_facts multiple
   times with different queries to gather information from each angle.
3. Only answer AFTER you have retrieved relevant facts from the tool.
4. Base your answer exclusively on the facts returned by the tool.
5. If the tool returns no relevant facts, say you couldn't find information.

Remember: searching is not optional. It is the entire point of this system."""

agente = create_agent(llm, [search_facts], prompt=SYSTEM_PROMPT)

print("✅ Agente RAG criado!")
print("   LLM: gemma4:e2b-it-qat")
print("   Ferramenta: search_facts (busca semântica no banco de fatos)")
print("   System prompt: SEMPRE buscar, NUNCA responder de memória")
import textwrap

def agente_stream(agente, pergunta, width=80):
    """Executa o agente com streaming — imprime cada passo em tempo real."""
    sep = "═" * min(width, 60)
    print(f"❓ {pergunta}")
    print(sep)

    for event in agente.stream({"messages": pergunta}, stream_mode="updates"):
        for node, data in event.items():
            for msg in data.get("messages", []):
                if msg.type == "human":
                    continue
                elif msg.type == "ai" and msg.tool_calls:
                    for tc in msg.tool_calls:
                        args_str = str(tc["args"])
                        if len(args_str) > width - 25:
                            args_str = args_str[:width - 28] + "..."
                        print(f"\n🤖 [busca] {tc['name']}({args_str})")
                elif msg.type == "tool":
                    content = msg.content[:200]
                    print("🔧 [resultado]")
                    for line in textwrap.wrap(content, width - 3):
                        print(f"   {line}")
                elif msg.type == "ai" and msg.content:
                    print("\n🤖 [resposta]")
                    for line in textwrap.wrap(msg.content, width):
                        print(line)
    print(f"\n{sep}")

## 6. Testando o agente

Vamos fazer uma pergunta que **exige combinar mais de um fato**.
O agente precisa buscar, recuperar, e conectar informações.

In [ ]:
# ── Teste 1: pergunta que conecta dois fatos ──
pergunta_agente = "Quem imaginou que máquinas poderiam pensar antes de os computadores existirem?"

agente_stream(agente, pergunta_agente)

In [ ]:
# ── Teste 2: Spinoza × Church-Turing (cognição × monismo) ──
# Força o agente a cruzar computação e metafísica — clusters distantes
pergunta_agente2 = "A Tese de Church-Turing sugere que processos cognitivos podem ser simulados por uma Máquina de Turing. Spinoza defende que mente e corpo são atributos da mesma substância. Essas visões sobre a mente são compatíveis? Busque fatos sobre ambos."

agente_stream(agente, pergunta_agente2)

---

## 📝 Resumo

- **Embeddings**: texto → vetor de números que captura significado
- **Similaridade por cosseno**: ângulo entre vetores = quão parecidos são
- **Matryoshka**: embeddings truncáveis (768 → 256 dim economiza memória)
- **FAISS**: vector store rápido e simples (create → add → search)
- **Busca semântica**: pergunta → embedding → FAISS → fatos relevantes
- **Agente RAG**: LLM + ferramenta de busca = buscar antes de responder
- **Combinação**: o agente pode buscar múltiplas vezes e conectar fatos

**Próxima aula:** Apresentação final — hora de mostrar os projetos!